# The Task: Pick Orange

## 📖 Reading only

The "Pick Orange" task ships with LeIsaac and is already installed at
`~/workspace/leisaac/source/leisaac/leisaac/tasks/pick_orange/`. 

## Task description

The robot must move **three oranges onto the plate, one at a time**, and then return to its rest
pose. Broken into subtasks, that's:

1. **Pick `Orange001`** — approach and grasp the first orange.
2. **Place it on the plate** — move it over the plate and open the gripper.
3. **Pick `Orange002`** → **place on plate**.
4. **Pick `Orange003`** → **place on plate**.
5. **Return to rest** — move back to the rest pose, which is what finally marks the task done.

Seven segments in total. Notice the robot isn't told *how* to do any of this — the task only
defines what counts as progress and what counts as finished. Producing the motion is the policy's
job.

## How progress is detected

`tasks/pick_orange/mdp/observations.py` defines two functions, each reused once per orange:

- **`orange_grasped(...)`** — true when the end effector is within `diff_threshold` (5 cm) of that
  orange **and** the gripper joint is closed (`joint_pos[:, -1] < 0.60`). Proximity alone isn't
  enough; the gripper has to actually be shut.
- **`put_orange_to_plate(...)`** — true when the orange's x/y position is inside the plate's
  `x_range`/`y_range` (±10 cm of the plate centre), the end effector is still near the orange, and
  the gripper is now **open**. The "still near, but open" pair is what distinguishes a deliberate
  release over the plate from an orange that was simply dropped.

Both are wired in as a `subtask_terms` observation group in `pick_orange_env_cfg.py`:

```python
class SubtaskCfg(ObsGroup):
    pick_orange001 = ObsTerm(func=mdp.orange_grasped, params={"object_cfg": SceneEntityCfg("Orange001")})
    put_orange001_to_plate = ObsTerm(
        func=mdp.put_orange_to_plate,
        params={"object_cfg": SceneEntityCfg("Orange001"), "plate_cfg": SceneEntityCfg("Plate")},
    )
    pick_orange002 = ObsTerm(func=mdp.orange_grasped, params={"object_cfg": SceneEntityCfg("Orange002")})
    put_orange002_to_plate = ObsTerm(...)
    pick_orange003 = ObsTerm(func=mdp.orange_grasped, params={"object_cfg": SceneEntityCfg("Orange003")})
    put_orange003_to_plate = ObsTerm(...)
```

These flags are **not** fed to the policy. They exist so the environment can tell where it is in
the task — used for success detection and, as you'll see below, for slicing demonstrations.

## What the policy actually sees

The `policy` observation group is the policy's whole world:

```python
joint_pos, joint_vel, joint_pos_rel, joint_vel_rel   # proprioception
actions                                              # the previous action
wrist, front                                         # two RGB camera images
ee_frame_state, joint_pos_target
```

Two cameras (`wrist` on the gripper, `front` on the base), 640×480 RGB at 30 FPS. That's why the
inference command in the next notebook uses `--enable_cameras` and why the GR00T server is started
with the **`so100_dualcam`** data config — "dual cam" is exactly this pair.

## How the episode ends

`tasks/pick_orange/mdp/terminations.py` defines `task_done(...)`, which requires **all three**
oranges to be within x/y/height range of the plate *and* the robot back at its rest pose. It's
registered alongside a timeout:

```python
class TerminationsCfg:
    time_out = DoneTerm(func=mdp.time_out, time_out=True)
    success = DoneTerm(func=mdp.task_done, params={
        "oranges_cfg": [SceneEntityCfg("Orange001"), SceneEntityCfg("Orange002"), SceneEntityCfg("Orange003")],
        "plate_cfg": SceneEntityCfg("Plate"),
    })
```

`episode_length_s = 8.0`, so an attempt that hasn't finished within 8 seconds is cut off and the
next round begins. This matters when you watch inference: a round ending is normal, not a crash.

## Putting it together: the environment config

`PickOrangeEnvCfg` in `pick_orange_env_cfg.py` combines everything — the `kitchen_with_orange`
scene from the previous notebook, the SO-101 follower arm, an `ee_frame` transformer that tracks
the gripper and jaw, the two cameras, a dome light, and the observation/termination groups above.

It also applies **domain randomization** on every reset:

```python
domain_randomization(self, random_options=[
    randomize_object_uniform("Orange001", pose_range={"x": (-0.03, 0.03), "y": (-0.03, 0.03), "z": (0.0, 0.0)}),
    randomize_object_uniform("Orange002", ...),
    randomize_object_uniform("Orange003", ...),
    randomize_object_uniform("Plate", ...),
    randomize_camera_uniform("front", pose_range={...}),  # ±2.5 cm, ±2.5°
])
```

Each orange and the plate get shifted up to ±3 cm, and the front camera is jittered slightly in
position and angle. A policy trained on this can't memorise one exact layout — it has to actually
look. It also means **each evaluation round you watch will be slightly different**, which is why the same policy can perform well in one round and poorly in the next.

## Generating training data: the mimic environment

`pick_orange_mimic_env_cfg.py` defines `PickOrangeMimicEnvCfg`, used only for **data generation** —
not for the inference you're about to run. It declares the same seven segments as `SubTaskConfig`
entries (`pick_orange001` → `put_orange001_to_plate` → ... → rest), each naming the object it
manipulates and the `subtask_term_signal` that marks its end:

```python
self.datagen_config.name = "pick_orange_leisaac_task_v0"

SubTaskConfig(
    object_ref="Orange001",
    subtask_term_signal="pick_orange001",
    selection_strategy="nearest_neighbor_object",
    action_noise=0.003,
    ...
)
```

With those boundaries declared, Isaac Lab Mimic can take a handful of human teleop demonstrations,
cut them at the subtask edges, and recombine and replay the segments against *new* object
placements with a little action noise — synthesising hundreds of training episodes from maybe ten
recorded by hand. This is how the fine-tuning dataset for the policy you're about to run was built.

## Registering the task

`tasks/pick_orange/__init__.py` registers both environments with Gymnasium so they can be
referenced by name:

```python
gym.register(id="LeIsaac-SO101-PickOrange-v0", ...)        # the one you'll run
gym.register(id="LeIsaac-SO101-PickOrange-Mimic-v0", ...)  # data generation only
```

You'll pass `LeIsaac-SO101-PickOrange-v0` to `--task` in the next notebook.